# **예측에 사용할 정보 확인**

모델은 예측 기준일 `t`에 이미 확인된 정보로 `t+1`~`t+7`의 판매량을 예측해야 합니다. 이 노트북은 과거 판매량, 같은 요일 판매, 달력과 매장·상품군 정보가 **어느 시점에 확보되는지** 보여줍니다.

가장 중요한 점검은 미래 판매량이 입력에 섞이지 않는지 확인하는 것입니다. 실제 Feature 생성은 `src/retail_forecast/`의 코드가 담당합니다.

> [그림 추가 자리: 기준일 `t`, 과거 lag, 1~7일 목표일의 시간축]

## **환경 준비**

Colab에서는 Drive를 연결합니다. 로컬과 Colab 모두 저장소 루트에서 실행합니다. 저장소 갱신은 별도의 00 노트북에서 마친 뒤 진행합니다.

In [ ]:
from pathlib import Path
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

REPO = Path.cwd()
REPO = REPO.parent if REPO.name == "notebooks" else REPO
assert (REPO / "pyproject.toml").is_file(), "저장소 루트에서 실행해 주세요."
os.chdir(REPO)

## **패키지 준비**

Feature 생성 코드와 그래프에 필요한 기존 프로젝트 패키지를 설치합니다.


In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e . --no-deps


## **Library**


In [ ]:
import pandas as pd

import seaborn as sns
import koreanize_matplotlib

from retail_forecast.config import load_config
from retail_forecast.features import get_feature_columns
from retail_forecast.pipeline import feature_cache_path, load_or_build_feature_table


## **Feature 선정 근거**

> **선정 원칙:** EDA에서 확인한 주간 반복·판매 규모 차이를 설명하고, 예측 기준일 `t`에 실제로 확보할 수 있는 값만 사용합니다. 목표일은 `d=t+h` (`h=1~7`)입니다.

| 구분 | 사용 Feature | 기준 날짜 | 채택 이유 |
|---|---|---|---|
| 최근 수요 | `sales`, `sales_lag_1/7/14/28/56` | `t`; `t−1`, `t−7`, `t−14`, `t−28`, `t−56` | 직전 판매 수준과 주간 반복을 함께 반영. 56일은 더 긴 주간 패턴을 확인하기 위한 후보 |
| 수요 안정화 | `sales_mean_7/14/28` | `t−1`까지의 최근 7·14·28개 관측값 | 하루 판매량의 일시적인 변동을 완화 |
| 동일 요일 | `sales_same_weekday_last_week`, `sales_same_weekday_4w_mean` | `d−7`, `d−14`, `d−21`, `d−28` | Baseline의 주간 패턴을 모델에도 제공. `h≤7`이라 기준일 이후 판매량을 사용하지 않음 |
| 예측 거리·달력 | `forecast_horizon`, 목표일의 요일·월·주차·주말·공휴일 | 목표일 `d` | 1~7일 거리와 미리 아는 날짜 효과를 구분 |
| 매장·상품군 | `store_nbr`, `family`, 도시·주·점포 유형·군집 | 기준일에 알려진 속성 | 매장·상품군별 판매 규모와 지역 차이를 반영 |
| 프로모션·거래량 | `onpromotion`, `current_transactions` | 기준일 `t` 영업 종료 시 | 그날의 판촉 규모와 거래 수준을 반영. 원본에 있는 `target_onpromotion`은 모델 입력에서 제외 |

**사용 조건:** `sales`와 기준일 거래량은 `t` 영업 종료 후 예측할 때만 사용할 수 있습니다. 목표일 판촉 수는 원본에 있지만 사전 확정 여부를 검증할 수 없어 입력에서 제외합니다.

**제외:** `target_sales`는 정답이고 `target_onpromotion`은 미래의 실제 판촉 수이므로 입력에서 제외합니다. 유가는 해석 가능성과 실효성이 낮아 사용하지 않습니다. 전체 이력 GPU 실험에서는 사전에 알 수 없는 지진·돌발 이벤트 변수도 제외합니다. 이 표는 사전 설계 근거이며, 각 변수의 실제 기여는 검증 점수와 중요도 분석으로 확인합니다.


In [ ]:
sns.set_theme(style="whitegrid", font="NanumGothic")

config = load_config(REPO / "configs" / "baseline.toml")
features = load_or_build_feature_table(config)
feature_columns = get_feature_columns(features)
feature_overview = pd.DataFrame([{"기준일 시작": features["date"].min().date(), "기준일 끝": features["date"].max().date(), "행 수": len(features), "Feature 캐시": str(feature_cache_path(config))}])
feature_overview

In [3]:
feature_summary = pd.DataFrame({
    "dtype": features[feature_columns].dtypes.astype(str),
    "missing": features[feature_columns].isna().sum(),
    "unique": features[feature_columns].nunique(),
})
feature_summary

,dtype,missing,unique
store_nbr,int16,0,54
family,category,0,33
sales,float32,0,118950
onpromotion,int16,0,265
sales_lag_1,float32,0,118937
sales_lag_7,float32,0,118854
sales_lag_14,float32,0,118768
sales_lag_28,float32,0,118623
sales_lag_56,float32,0,118230
sales_mean_7,float32,0,157569


## **생성 결과 예시**

In [ ]:
example_columns = [
    "date", "target_date", "forecast_horizon", "store_nbr", "family",
    "sales", "sales_lag_7", "sales_lag_28", "sales_mean_7",
    "sales_same_weekday_last_week", "sales_same_weekday_4w_mean",
    "target_onpromotion", "target_dayofweek", "target_sales",
]
feature_example = features[example_columns].sample(10, random_state=42).sort_values(["store_nbr", "family", "target_date"])
feature_example


## **데이터 누수 점검**

- `target_sales`는 X 변수에서 제외합니다.
- 판매 lag와 이동평균은 기준일 또는 그 이전 값으로만 계산합니다.
- 목표일 정보는 사전에 정해지는 달력 정보만 사용합니다. 미래의 실제 판촉 수는 입력에서 제외합니다.
- 거래량은 목표일 값이 아니라 예측 기준일의 관측값을 사용합니다.
- 유가는 일별 수요와의 관계를 명확히 설명하기 어려워 제외합니다.
- 검증 시작일 이후에 관측되는 정답은 학습에서 제외합니다.

In [ ]:
leakage_checks = pd.Series({
    "target_is_excluded": "target_sales" not in feature_columns,
    "future_promotion_is_excluded": "target_onpromotion" not in feature_columns,
    "horizon_matches_dates": features["target_date"].sub(features["date"]).dt.days.eq(features["forecast_horizon"]).all(),
    "horizons_are_one_to_seven": set(features["forecast_horizon"].unique()) == set(range(1, 8)),
    "weekly_reference_is_known": features["seasonal_reference_date"].le(features["date"]).all(),
    "identifier_is_excluded": "id" not in feature_columns,
    "target_is_non_negative": features["target_sales"].ge(0).all(),
})
leakage_checks

`sales`는 라벨이 아니라 예측 기준일에 관측한 판매량이고, `target_sales`가 각 `forecast_horizon`의 실제 라벨입니다. 동일한 기준일이 1~7일의 행으로 확장되며, 모델은 과거 판매 변수와 목표일의 사전 확정 정보를 함께 학습합니다.